# 02 Initial Model Experiments

Leakage-safe baseline ve initial model karsilastirmasi.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.evaluation import evaluate_classifier
from src.modeling import build_pipeline, get_model_specs
from src.preprocessing import (
    build_preprocessor,
    get_feature_groups,
    split_features_target,
    validate_training_data,
)
from src.project_paths import INCOME_CSV, OUTPUTS_DIR, REPORTS_DIR

RANDOM_STATE = 42
OUTPUTS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

In [ ]:
train_df = pd.read_csv(INCOME_CSV)
validate_training_data(train_df)
X, y = split_features_target(train_df)
features = get_feature_groups()
selected_features = list(features.all_features)

X_train, X_validation, y_train, y_validation = train_test_split(
    X[selected_features],
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'train split: {X_train.shape}, validation split: {X_validation.shape}')

In [ ]:
rows = []
for spec in get_model_specs(random_state=RANDOM_STATE):
    preprocessor = build_preprocessor(features.numeric, features.categorical)
    pipeline = build_pipeline(preprocessor, spec.estimator)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_classifier(
        pipeline,
        X_train,
        y_train,
        split_name='train',
        model_name=spec.name,
        variant='full',
    )
    validation_metrics = evaluate_classifier(
        pipeline,
        X_validation,
        y_validation,
        split_name='validation',
        model_name=spec.name,
        variant='full',
    )
    rows.extend([train_metrics, validation_metrics])

model_comparison = pd.concat(rows, ignore_index=True)
model_comparison.to_csv(OUTPUTS_DIR / 'model_comparison.csv', index=False)
model_comparison

In [ ]:
validation_results = (
    model_comparison[model_comparison['split'] == 'validation']
    .sort_values('auc', ascending=False)
    .reset_index(drop=True)
)
validation_results

In [ ]:
best = validation_results.iloc[0]
notes_path = REPORTS_DIR / 'report_notes_tr.md'
existing = notes_path.read_text(encoding='utf-8') if notes_path.exists() else '# Rolling Report Notes\n\n'
notes = f'''## T5 - Initial model comparison

- Initial model deneylerinde Logistic Regression, Random Forest ve HistGradientBoosting modelleri ayni full feature set ile karsilastirildi.
- Birincil metrik AUC olarak tutuldu; en yuksek validation AUC {best['model']} modelinde {best['auc']:.3f} olarak goruldu.
- Bu sonuclar final model secimi degildir; T6-T8 tuning, ablation ve fairness analizleri sonrasi karar verilecektir.
- `income_test.csv` bu asamada model secimi veya threshold belirlemek icin kullanilmadi.

'''
notes_path.write_text(existing.rstrip() + '\n\n' + notes, encoding='utf-8')
print(notes)